<a href="https://colab.research.google.com/github/NzizaPacifique250/Deep-Q-Learning-A3/blob/david/YinkaAjao_DQN_formative3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

project_path = '/content/drive/MyDrive/Formative_3_DQN_Atari'

# Create the folder if it doesn't exist
os.makedirs(project_path, exist_ok=True)

os.chdir(project_path)

print(f"Success! Current working directory is now: {os.getcwd()}")

Success! Current working directory is now: /content/drive/MyDrive/Formative_3_DQN_Atari


In [ ]:
# Force remove conflicting default packages
!pip uninstall -y gym gymnasium stable-baselines3 ale-py autorom > /dev/null 2>&1

# Install system dependencies for the headless display
!apt-get update > /dev/null 2>&1
!apt-get install -y xvfb python3-opengl ffmpeg > /dev/null 2>&1

# Install the strict compatible versions requested by your group
!pip install -q "gymnasium[atari,accept-rom-license]>=0.29.0" "stable-baselines3[extra]>=2.3.0" "ale-py>=0.9.0" opencv-python pyvirtualdisplay tensorboard
!AutoROM --accept-license > /dev/null 2>&1

print("Dependencies installed successfully! Please RESTART SESSION now (Run > Restart Session) before running Cell 2.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, which is not installed.
Dependencies installed successfully! Please RESTART SESSION now (Run > Restart Session) before running Cell 2.


In [ ]:
%%writefile train.py
"""
train.py — DQN training skeleton for the Atari group project.
"""

import argparse
import os
import gymnasium as gym
import ale_py
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

# Register ALE environments
gym.register_envs(ale_py)


# Group-agreed constants
ENV_ID = "ALE/SpaceInvaders-v5"
N_ENVS = 4
FRAME_STACK = 4
POLICY = "CnnPolicy"
LOG_DIR = "./tb_logs"
MODEL_DIR = "./models"


def parse_args():
    parser = argparse.ArgumentParser(description="Train a DQN agent on an Atari environment.")
    parser.add_argument("--lr", type=float, default=1e-4, help="Learning rate")
    parser.add_argument("--gamma", type=float, default=0.99, help="Discount factor")
    parser.add_argument("--batch-size", type=int, default=32, help="Batch size")
    parser.add_argument("--eps-start", type=float, default=1.0, help="Initial epsilon (exploration_initial_eps)")
    parser.add_argument("--eps-end", type=float, default=0.05, help="Final epsilon (exploration_final_eps)")
    parser.add_argument("--eps-decay-frac", type=float, default=0.1, help="Fraction of training over which epsilon decays (exploration_fraction)")
    parser.add_argument("--timesteps", type=int, default=200_000, help="Total training timesteps")
    parser.add_argument("--policy", type=str, default=POLICY, choices=["CnnPolicy", "MlpPolicy"], help="Policy network type")
    parser.add_argument("--run-name", type=str, default="run", help="Name used for logs/model file")
    return parser.parse_args()


def make_env():
    """Build a vectorized, frame-stacked Atari environment."""
    env = make_atari_env(ENV_ID, n_envs=N_ENVS, seed=0)
    env = VecFrameStack(env, n_stack=FRAME_STACK)
    return env


def main():
    args = parse_args()
    os.makedirs(MODEL_DIR, exist_ok=True)
    os.makedirs(LOG_DIR, exist_ok=True)

    env = make_env()

    model = DQN(
        policy=args.policy,
        env=env,
        learning_rate=args.lr,
        gamma=args.gamma,
        batch_size=args.batch_size,
        exploration_initial_eps=args.eps_start,
        exploration_final_eps=args.eps_end,
        exploration_fraction=args.eps_decay_frac,
        buffer_size=100_000,
        learning_starts=10_000,
        train_freq=4,
        target_update_interval=1000,
        verbose=1,
        tensorboard_log=LOG_DIR,
        device="cuda" # Force GPU usage in Colab
    )

    print(f"Training config: {vars(args)}")
    model.learn(
        total_timesteps=args.timesteps,
        tb_log_name=args.run_name,
        progress_bar=True,
    )

    save_path = os.path.join(MODEL_DIR, f"dqn_model_{args.run_name}.zip")
    model.save(save_path)
    print(f"Saved model to {save_path}")

    env.close()

if __name__ == "__main__":
    main()

Overwriting train.py


In [ ]:
!python train.py

Streaming output truncated to the last 5000 lines.
|    n_updates        | 7678     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 696      |
|    ep_rew_mean      | 297      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 2612     |
|    fps              | 92       |
|    time_elapsed     | 1437     |
|    total_timesteps  | 133024   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.132    |
|    n_updates        | 7688     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 691      |
|    ep_rew_mean      | 290      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 2616     |
|    fps              | 92       |
|    time_elapsed     | 1439     |
|    total_timesteps  | 133188   |
| tr

In [ ]:
%%writefile play.py
"""
play.py — load a trained DQN model and watch it play, greedily.
FAST EVALUATION MODE (Video Disabled)
"""

import argparse
import os
import gymnasium as gym
import ale_py
from stable_baselines3 import DQN

# Register environments
gym.register_envs(ale_py)

ENV_ID = "ALE/SpaceInvaders-v5"
FRAME_STACK = 4

def parse_args():
    parser = argparse.ArgumentParser(description="Play an Atari environment with a trained DQN model.")
    parser.add_argument("--model", type=str, default="models/dqn_model_exp01.zip", help="Path to the saved model .zip")
    parser.add_argument("--episodes", type=int, default=3, help="Number of episodes to play")
    return parser.parse_args()

def main():
    args = parse_args()

    print(f"Loading model from {args.model}...")

    # Recreate the exact environment wrapping with frameskip=1
    env = gym.make(ENV_ID, render_mode="rgb_array", frameskip=1)
    env = gym.wrappers.AtariPreprocessing(env)
    env = gym.wrappers.FrameStackObservation(env, FRAME_STACK)

    model = DQN.load(args.model)

    for ep in range(1, args.episodes + 1):
        obs, info = env.reset()
        done = False
        total_reward = 0.0
        steps = 0

        while not done:
            action, _states = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            steps += 1
            done = terminated or truncated

        print(f"Episode {ep}: reward={total_reward:.1f}, steps={steps}")

    env.close()

if __name__ == "__main__":
    main()

Overwriting play.py


In [ ]:
%%writefile run_experiments.py
"""
run_experiments.py — run one member's 10 hyperparameter experiments in sequence.

Each experiment is a distinct combination of (lr, gamma, batch_size, eps_*),
trained by calling train.py as a subprocess so every run gets its own logs,
monitor CSVs, best model, and a summary row in experiments/results.csv.
"""

import argparse
import subprocess
import sys

MEMBER = "Yinka_Ajao"

# Aligned with Edwin's sweep layout: modifying ONE variable at a time from baseline
EXPERIMENTS = [
    # id,  lr,     gamma, batch, eps_start, eps_end, eps_decay_frac
    (1,  1e-4,  0.99,  32,   1.0,  0.05, 0.10),   # Baseline config
    (2,  1e-3,  0.99,  32,   1.0,  0.05, 0.10),   # Higher learning rate
    (3,  1e-5,  0.99,  32,   1.0,  0.05, 0.10),   # Lower learning rate
    (4,  1e-4,  0.90,  32,   1.0,  0.05, 0.10),   # Shorter horizon (low gamma)
    (5,  1e-4,  0.995, 32,   1.0,  0.05, 0.10),   # Longer horizon (high gamma)
    (6,  1e-4,  0.99,  16,   1.0,  0.05, 0.10),   # Smaller batch size
    (7,  1e-4,  0.99,  128,  1.0,  0.05, 0.10),   # Larger batch size
    (8,  1e-4,  0.99,  32,   1.0,  0.05, 0.02),   # Rapid epsilon decay
    (9,  1e-4,  0.99,  32,   1.0,  0.10, 0.30),   # Prolonged exploration
    (10, 5e-4,  0.99,  64,   1.0,  0.02, 0.15),   # Combined optimal guess config
]

def parse_args():
    p = argparse.ArgumentParser(description="Run a member's 10 DQN hyperparameter experiments.")
    p.add_argument("--member", type=str, default=MEMBER, help="Member name used in run-name prefix")
    p.add_argument("--timesteps", type=int, default=150_000, help="Timesteps per experiment")
    p.add_argument("--policy", type=str, default="CnnPolicy", choices=["CnnPolicy", "MlpPolicy"])
    p.add_argument("--only", type=int, nargs="*", default=None, help="Only run these experiment ids")
    return p.parse_args()

def main():
    args = parse_args()
    to_run = [e for e in EXPERIMENTS if args.only is None or e[0] in args.only]
    print(f"Running {len(to_run)} experiment(s) for member '{args.member}' "
          f"at {args.timesteps} timesteps each ({args.policy}).\n")

    for (eid, lr, gamma, batch, es, ee, ed) in to_run:
        run_name = f"{args.member}_exp{eid:02d}"
        cmd = [
            sys.executable, "train.py",
            "--lr", str(lr),
            "--gamma", str(gamma),
            "--batch-size", str(batch),
            "--eps-start", str(es),
            "--eps-end", str(ee),
            "--eps-decay-frac", str(ed),
            "--timesteps", str(args.timesteps),
            "--policy", args.policy,
            "--run-name", run_name,
        ]
        print("=" * 70)
        print(f"[{eid}/10] {run_name}: lr={lr} gamma={gamma} batch={batch} "
              f"eps={es}->{ee} decay={ed}")
        print("=" * 70)
        result = subprocess.run(cmd)
        if result.returncode != 0:
            print(f"!! Experiment {eid} failed (exit {result.returncode}). Continuing.\n")

    print("\nAll requested experiments finished.")

if __name__ == "__main__":
    main()

Overwriting run_experiments.py


In [ ]:
!python run_experiments.py --timesteps 150000 --only 1 2 3 4 5

Streaming output truncated to the last 5000 lines.
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 650      |
|    ep_rew_mean      | 245      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1696     |
|    fps              | 88       |
|    time_elapsed     | 927      |
|    total_timesteps  | 82092    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.44     |
|    n_updates        | 4505     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 652      |
|    ep_rew_mean      | 245      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1700     |
|    fps              | 88       |
|    time_elapsed     | 929      |
|    total_timesteps  | 82336    |
| train/              |          |
|   

In [ ]:
!python run_experiments.py --timesteps 150000 --only 6 7 8 9 10

Streaming output truncated to the last 5000 lines.
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 571      |
|    ep_rew_mean      | 216      |
|    exploration_rate | 0.02     |
| time/               |          |
|    episodes         | 1820     |
|    fps              | 64       |
|    time_elapsed     | 1273     |
|    total_timesteps  | 81920    |
| train/              |          |
|    learning_rate    | 0.0005   |
|    loss             | 0.119    |
|    n_updates        | 4494     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 571      |
|    ep_rew_mean      | 216      |
|    exploration_rate | 0.02     |
| time/               |          |
|    episodes         | 1824     |
|    fps              | 64       |
|    time_elapsed     | 1275     |
|    total_timesteps  | 82080    |
| train/              |          |
|   

In [ ]:
 # Play experiment 1 (original model)
!python play.py --model models/dqn_model_Yinka_Ajao_exp01.zip --episodes 3

2026-07-19 14:59:34.675794: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-19 14:59:34.802904: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading model from models/dqn_model_Yinka_Ajao_exp01.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=290.0, steps=869
Episode 2: reward=90.0, steps=470
Episode 3: reward=60.0, steps=388


In [ ]:
# Play experiment 2 (original model)
!python play.py --model models/dqn_model_Yinka_Ajao_exp02.zip --episodes 3

2026-07-19 14:59:53.610923: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-19 14:59:53.683706: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading model from models/dqn_model_Yinka_Ajao_exp02.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=40.0, steps=379
Episode 2: reward=80.0, steps=367
Episode 3: reward=630.0, steps=857


In [ ]:
# Play experiment 3 (original model)
!python play.py --model models/dqn_model_Yinka_Ajao_exp03.zip --episodes 3

2026-07-19 15:00:30.044751: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-19 15:00:30.159820: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading model from models/dqn_model_Yinka_Ajao_exp03.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=285.0, steps=725
Episode 2: reward=285.0, steps=721
Episode 3: reward=285.0, steps=726


In [ ]:
# Play experiment 4 (original model)
!python play.py --model models/dqn_model_Yinka_Ajao_exp04.zip --episodes 3

2026-07-19 15:00:50.053768: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-19 15:00:50.126868: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading model from models/dqn_model_Yinka_Ajao_exp04.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=145.0, steps=942
Episode 2: reward=80.0, steps=510
Episode 3: reward=135.0, steps=509


In [ ]:
# Play experiment 5 (original model)
!python play.py --model models/dqn_model_Yinka_Ajao_exp05.zip --episodes 3

2026-07-19 15:01:13.808381: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-19 15:01:13.930263: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading model from models/dqn_model_Yinka_Ajao_exp05.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=130.0, steps=509
Episode 2: reward=130.0, steps=507
Episode 3: reward=260.0, steps=745


In [ ]:
# Play experiment 6 (original model)
!python play.py --model models/dqn_model_Yinka_Ajao_exp06.zip --episodes 3

2026-07-19 15:01:33.984215: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-19 15:01:34.059460: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading model from models/dqn_model_Yinka_Ajao_exp06.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=30.0, steps=288
Episode 2: reward=45.0, steps=509
Episode 3: reward=45.0, steps=475


In [ ]:
# Play experiment 7 (original model)
!python play.py --model models/dqn_model_Yinka_Ajao_exp07.zip --episodes 3

2026-07-19 15:01:50.345714: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-19 15:01:50.417705: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading model from models/dqn_model_Yinka_Ajao_exp07.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=215.0, steps=710
Episode 2: reward=350.0, steps=720
Episode 3: reward=340.0, steps=795


In [ ]:
# Play experiment 8 (original model)
!python play.py --model models/dqn_model_Yinka_Ajao_exp08.zip --episodes 3

2026-07-19 15:02:09.864162: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-19 15:02:09.938356: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading model from models/dqn_model_Yinka_Ajao_exp08.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=215.0, steps=848
Episode 2: reward=215.0, steps=835
Episode 3: reward=5.0, steps=240


In [ ]:
# Play experiment 9 (original model)
!python play.py --model models/dqn_model_Yinka_Ajao_exp09.zip --episodes 3

2026-07-19 15:02:28.690511: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-19 15:02:28.807466: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading model from models/dqn_model_Yinka_Ajao_exp09.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=75.0, steps=461
Episode 2: reward=210.0, steps=466
Episode 3: reward=135.0, steps=517


In [ ]:
# Play experiment 10 (original model)
!python play.py --model models/dqn_model_Yinka_Ajao_exp10.zip --episodes 3

2026-07-19 15:02:47.040117: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-19 15:02:47.158214: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading model from models/dqn_model_Yinka_Ajao_exp10.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=120.0, steps=352
Episode 2: reward=655.0, steps=817
Episode 3: reward=110.0, steps=480


In [ ]:
# Run experiment 1
!python run_experiments.py --timesteps 150000 --only 1 --policy MlpPolicy --member Yinka_Ajao_MLP

Streaming output truncated to the last 5000 lines.
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 606      |
|    ep_rew_mean      | 225      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1836     |
|    fps              | 129      |
|    time_elapsed     | 670      |
|    total_timesteps  | 86988    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.257    |
|    n_updates        | 4811     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 604      |
|    ep_rew_mean      | 221      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1840     |
|    fps              | 129      |
|    time_elapsed     | 673      |
|    total_timesteps  | 87348    |
| train/              |          |
|   

In [ ]:
# Run experiment 2
!python run_experiments.py --timesteps 150000 --only 2 --policy MlpPolicy --member Yinka_Ajao_MLP

Streaming output truncated to the last 5000 lines.
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 639      |
|    ep_rew_mean      | 233      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1744     |
|    fps              | 127      |
|    time_elapsed     | 662      |
|    total_timesteps  | 84736    |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.117    |
|    n_updates        | 4670     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 637      |
|    ep_rew_mean      | 232      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1748     |
|    fps              | 127      |
|    time_elapsed     | 664      |
|    total_timesteps  | 84904    |
| train/              |          |
|   

In [ ]:
# Run experiment 3
!python run_experiments.py --timesteps 150000 --only 3 --policy MlpPolicy --member Yinka_Ajao_MLP

Streaming output truncated to the last 5000 lines.
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 645      |
|    ep_rew_mean      | 246      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1968     |
|    fps              | 138      |
|    time_elapsed     | 633      |
|    total_timesteps  | 87776    |
| train/              |          |
|    learning_rate    | 1e-05    |
|    loss             | 0.233    |
|    n_updates        | 4860     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 643      |
|    ep_rew_mean      | 244      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1972     |
|    fps              | 138      |
|    time_elapsed     | 635      |
|    total_timesteps  | 87984    |
| train/              |          |
|   

In [ ]:
# Run experiment 4
!python run_experiments.py --timesteps 150000 --only 4 --policy MlpPolicy --member Yinka_Ajao_MLP

Streaming output truncated to the last 5000 lines.
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 616      |
|    ep_rew_mean      | 239      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1888     |
|    fps              | 131      |
|    time_elapsed     | 653      |
|    total_timesteps  | 86128    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0385   |
|    n_updates        | 4757     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 614      |
|    ep_rew_mean      | 238      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1892     |
|    fps              | 131      |
|    time_elapsed     | 654      |
|    total_timesteps  | 86224    |
| train/              |          |
|   

In [ ]:
# Run experiment 5
!python run_experiments.py --timesteps 150000 --only 5 --policy MlpPolicy --member Yinka_Ajao_MLP

Streaming output truncated to the last 5000 lines.
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 653      |
|    ep_rew_mean      | 251      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1836     |
|    fps              | 128      |
|    time_elapsed     | 668      |
|    total_timesteps  | 86008    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0984   |
|    n_updates        | 4750     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 650      |
|    ep_rew_mean      | 250      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1840     |
|    fps              | 128      |
|    time_elapsed     | 670      |
|    total_timesteps  | 86264    |
| train/              |          |
|   

In [ ]:
# Run experiment 6
!python run_experiments.py --timesteps 150000 --only 6 --policy MlpPolicy --member Yinka_Ajao_MLP

Streaming output truncated to the last 5000 lines.
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 593      |
|    ep_rew_mean      | 210      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1808     |
|    fps              | 137      |
|    time_elapsed     | 619      |
|    total_timesteps  | 84972    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.939    |
|    n_updates        | 4685     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 595      |
|    ep_rew_mean      | 210      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1812     |
|    fps              | 137      |
|    time_elapsed     | 619      |
|    total_timesteps  | 85048    |
| train/              |          |
|   

In [ ]:
# Run experiment 7
!python run_experiments.py --timesteps 150000 --only 7 --policy MlpPolicy --member Yinka_Ajao_MLP

Streaming output truncated to the last 5000 lines.
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 572      |
|    ep_rew_mean      | 207      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1884     |
|    fps              | 92       |
|    time_elapsed     | 937      |
|    total_timesteps  | 86700    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.106    |
|    n_updates        | 4793     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 570      |
|    ep_rew_mean      | 207      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1888     |
|    fps              | 92       |
|    time_elapsed     | 940      |
|    total_timesteps  | 86920    |
| train/              |          |
|   

In [ ]:
# Run experiment 8
!python run_experiments.py --timesteps 150000 --only 8 --policy MlpPolicy --member Yinka_Ajao_MLP

Streaming output truncated to the last 5000 lines.
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 610      |
|    ep_rew_mean      | 236      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1880     |
|    fps              | 129      |
|    time_elapsed     | 671      |
|    total_timesteps  | 87084    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.409    |
|    n_updates        | 4817     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 618      |
|    ep_rew_mean      | 240      |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1884     |
|    fps              | 129      |
|    time_elapsed     | 673      |
|    total_timesteps  | 87328    |
| train/              |          |
|   

In [ ]:
# Run experiment 9
!python run_experiments.py --timesteps 150000 --only 9 --policy MlpPolicy --member Yinka_Ajao_MLP

Streaming output truncated to the last 5000 lines.
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 643      |
|    ep_rew_mean      | 250      |
|    exploration_rate | 0.1      |
| time/               |          |
|    episodes         | 1904     |
|    fps              | 137      |
|    time_elapsed     | 621      |
|    total_timesteps  | 85264    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.215    |
|    n_updates        | 4703     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 646      |
|    ep_rew_mean      | 251      |
|    exploration_rate | 0.1      |
| time/               |          |
|    episodes         | 1908     |
|    fps              | 137      |
|    time_elapsed     | 622      |
|    total_timesteps  | 85472    |
| train/              |          |
|   

In [ ]:
# Run experiment 10
!python run_experiments.py --timesteps 150000 --only 10 --policy MlpPolicy --member Yinka_Ajao_MLP

Streaming output truncated to the last 5000 lines.
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 662      |
|    ep_rew_mean      | 252      |
|    exploration_rate | 0.02     |
| time/               |          |
|    episodes         | 1744     |
|    fps              | 117      |
|    time_elapsed     | 696      |
|    total_timesteps  | 82084    |
| train/              |          |
|    learning_rate    | 0.0005   |
|    loss             | 0.217    |
|    n_updates        | 4505     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 658      |
|    ep_rew_mean      | 250      |
|    exploration_rate | 0.02     |
| time/               |          |
|    episodes         | 1748     |
|    fps              | 117      |
|    time_elapsed     | 698      |
|    total_timesteps  | 82228    |
| train/              |          |
|   

In [ ]:
# Play experiment 1
!python play.py --model models/dqn_model_Yinka_Ajao_MLP_exp01.zip --episodes 3

Loading model from models/dqn_model_Yinka_Ajao_MLP_exp01.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=165.0, steps=682
Episode 2: reward=80.0, steps=377
Episode 3: reward=310.0, steps=603


In [ ]:
# Play experiment 2
!python play.py --model models/dqn_model_Yinka_Ajao_MLP_exp02.zip --episodes 3

Loading model from models/dqn_model_Yinka_Ajao_MLP_exp02.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=325.0, steps=794
Episode 2: reward=155.0, steps=518
Episode 3: reward=250.0, steps=611


In [ ]:
# Play experiment 3
!python play.py --model models/dqn_model_Yinka_Ajao_MLP_exp03.zip --episodes 3

Loading model from models/dqn_model_Yinka_Ajao_MLP_exp03.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=340.0, steps=798
Episode 2: reward=285.0, steps=748
Episode 3: reward=245.0, steps=607


In [ ]:
# Play experiment 4
!python play.py --model models/dqn_model_Yinka_Ajao_MLP_exp04.zip --episodes 3

Loading model from models/dqn_model_Yinka_Ajao_MLP_exp04.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=170.0, steps=379
Episode 2: reward=365.0, steps=555
Episode 3: reward=225.0, steps=463


In [ ]:
# Play experiment 5
!python play.py --model models/dqn_model_Yinka_Ajao_MLP_exp05.zip --episodes 3

Loading model from models/dqn_model_Yinka_Ajao_MLP_exp05.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=5.0, steps=277
Episode 2: reward=0.0, steps=241
Episode 3: reward=5.0, steps=383


In [ ]:
# Play experiment 6
!python play.py --model models/dqn_model_Yinka_Ajao_MLP_exp06.zip --episodes 3

Loading model from models/dqn_model_Yinka_Ajao_MLP_exp06.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=525.0, steps=847
Episode 2: reward=160.0, steps=507
Episode 3: reward=315.0, steps=871


In [ ]:
# Play experiment 7
!python play.py --model models/dqn_model_Yinka_Ajao_MLP_exp07.zip --episodes 3

Loading model from models/dqn_model_Yinka_Ajao_MLP_exp07.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=245.0, steps=582
Episode 2: reward=235.0, steps=647
Episode 3: reward=275.0, steps=799


In [ ]:
# Play experiment 8
!python play.py --model models/dqn_model_Yinka_Ajao_MLP_exp08.zip --episodes 3

Loading model from models/dqn_model_Yinka_Ajao_MLP_exp08.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=350.0, steps=833
Episode 2: reward=520.0, steps=931
Episode 3: reward=185.0, steps=487


In [ ]:
# Play experiment 9
!python play.py --model models/dqn_model_Yinka_Ajao_MLP_exp09.zip --episodes 3

Loading model from models/dqn_model_Yinka_Ajao_MLP_exp09.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=320.0, steps=664
Episode 2: reward=100.0, steps=375
Episode 3: reward=390.0, steps=681


In [ ]:
# Play experiment 10
!python play.py --model models/dqn_model_Yinka_Ajao_MLP_exp10.zip --episodes 3

Loading model from models/dqn_model_Yinka_Ajao_MLP_exp10.zip...
A.L.E: Arcade Learning Environment (version 0.12.0+0706845)
[Powered by Stella]
Episode 1: reward=230.0, steps=597
Episode 2: reward=190.0, steps=701
Episode 3: reward=325.0, steps=967


In [ ]:
"""
automate_rubric.py — Evaluate all models, crown the champion, and record the video.
"""

import glob
import os
import shutil
import numpy as np
import gymnasium as gym
import ale_py
from stable_baselines3 import DQN
from gymnasium.wrappers import RecordVideo
from pyvirtualdisplay import Display

# Config
MEMBER_NAME = "Yinka_Ajao"
ENV_ID = "ALE/SpaceInvaders-v5"
FRAME_STACK = 4
EPISODES_TO_EVAL = 3

# Register environments
gym.register_envs(ale_py)

def evaluate_model(model_path, env, episodes=3):
    """Loads a model, plays N episodes, and returns the average reward."""
    model = DQN.load(model_path)
    rewards = []

    for _ in range(episodes):
        obs, info = env.reset()
        done = False
        total_reward = 0.0
        while not done:
            action, _states = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated
        rewards.append(total_reward)

    return np.mean(rewards)

def main():
    print("STEP 1: Evaluating all experiments to generate table data...")

    # Fast environment without video recording
    eval_env = gym.make(ENV_ID, render_mode="rgb_array", frameskip=1)
    eval_env = gym.wrappers.AtariPreprocessing(eval_env)
    eval_env = gym.wrappers.FrameStackObservation(eval_env, FRAME_STACK)

    # Search for both standard and MLP models
    model_files = glob.glob(f"models/dqn_model_{MEMBER_NAME}_exp*.zip")
    model_files.extend(glob.glob(f"models/dqn_model_{MEMBER_NAME}_MLP_exp*.zip"))

    if not model_files:
        print("No models found! Check your MEMBER_NAME or folder path.")
        return

    best_model_path = None
    best_avg_reward = -float('inf')

    # Markdown table header for easy copy-pasting
    print("\n| Experiment File | Avg Reward (3 eps) |")
    print("| :--- | :--- |")

    for mf in sorted(model_files):
        avg_score = evaluate_model(mf, eval_env, EPISODES_TO_EVAL)
        filename = os.path.basename(mf)
        print(f"| {filename} | {avg_score:.1f} |")

        if avg_score > best_avg_reward:
            best_avg_reward = avg_score
            best_model_path = mf

    eval_env.close()

    print("\nSTEP 2: Crowning the Champion...")
    print(f"Best Agent: {os.path.basename(best_model_path)} with {best_avg_reward:.1f} avg reward.")

    # Copy instead of rename, so the original experiment file is preserved
    final_model_name = "dqn_model.zip"
    shutil.copy(best_model_path, final_model_name)
    print(f"Copied winning model to ./{final_model_name}")

    print("\nSTEP 3: Recording the Highlight Reel...")
    # Start the virtual display for video rendering
    virtual_display = Display(visible=0, size=(1400, 900))
    virtual_display.start()

    # Create a new environment wrapped with RecordVideo
    rec_env = gym.make(ENV_ID, render_mode="rgb_array", frameskip=1)
    rec_env = gym.wrappers.AtariPreprocessing(rec_env)
    rec_env = gym.wrappers.FrameStackObservation(rec_env, FRAME_STACK)

    video_dir = "./video"
    os.makedirs(video_dir, exist_ok=True)
    rec_env = RecordVideo(rec_env, video_folder=video_dir, name_prefix="champion-video")

    # Evaluate the champion for 1 episode just to get the footage
    evaluate_model(final_model_name, rec_env, episodes=1)

    rec_env.close()
    virtual_display.stop()
    print("Done!, Champion's video is in the ./video folder.")

if __name__ == "__main__":
    main()

STEP 1: Evaluating all experiments to generate table data...

| Experiment File | Avg Reward (3 eps) |
| :--- | :--- |
| dqn_model_Yinka_Ajao_MLP_exp01.zip | 243.3 |
| dqn_model_Yinka_Ajao_MLP_exp02.zip | 188.3 |
| dqn_model_Yinka_Ajao_MLP_exp03.zip | 170.0 |
| dqn_model_Yinka_Ajao_MLP_exp04.zip | 185.0 |
| dqn_model_Yinka_Ajao_MLP_exp05.zip | 3.3 |
| dqn_model_Yinka_Ajao_MLP_exp06.zip | 310.0 |
| dqn_model_Yinka_Ajao_MLP_exp07.zip | 146.7 |
| dqn_model_Yinka_Ajao_MLP_exp08.zip | 413.3 |
| dqn_model_Yinka_Ajao_MLP_exp09.zip | 295.0 |
| dqn_model_Yinka_Ajao_MLP_exp10.zip | 181.7 |
| dqn_model_Yinka_Ajao_exp01.zip | 310.0 |
| dqn_model_Yinka_Ajao_exp02.zip | 233.3 |
| dqn_model_Yinka_Ajao_exp03.zip | 285.0 |
| dqn_model_Yinka_Ajao_exp04.zip | 188.3 |
| dqn_model_Yinka_Ajao_exp05.zip | 110.0 |
| dqn_model_Yinka_Ajao_exp06.zip | 111.7 |
| dqn_model_Yinka_Ajao_exp07.zip | 151.7 |
| dqn_model_Yinka_Ajao_exp08.zip | 158.3 |
| dqn_model_Yinka_Ajao_exp09.zip | 263.3 |
| dqn_model_Yinka_Ajao_exp

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content/drive/MyDrive/Formative_3_DQN_Atari/video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Done!, Champion's video is in the ./video folder.
